In [1]:
df=spark.sql("SELECT * from tbl_latest_news")
display(df)

StatementMeta(, 7e76f964-dd43-4a1f-87fc-8e380ab5258e, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6548505b-c2d7-4984-9f5f-7aec8a7192bb)

In [2]:
from synapse.ml.spark.aifunc.DataFrameExtensions import AIFunctions

sentiment_df = df.ai.analyze_sentiment(input_col="snippet", output_col="sentiment")
display(sentiment_df)

StatementMeta(, 7e76f964-dd43-4a1f-87fc-8e380ab5258e, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f7a15ec2-1810-4cda-96c0-6efc1c638889)

In [3]:
#using type 1
from pyspark.sql.utils import AnalysisException

try:
    table_name = "tbl_sentiment_analysis"

    sentiment_df.write.format("delta").saveAsTable(table_name)

except AnalysisException:
    print("Table already exists")

    sentiment_df.createOrReplaceTempView("vw_df_sentiment")

    spark.sql(f"""
        MERGE INTO {table_name} target_table
        USING vw_df_sentiment source_view
        ON source_view.link = target_table.link

        WHEN MATCHED AND (
            source_view.title <> target_table.title OR
            source_view.snippet <> target_table.snippet OR
            source_view.date <> target_table.date OR
            source_view.favicon <> target_table.favicon OR
            source_view.source <> target_table.source OR
            source_view.position <> target_table.position
        )
        THEN UPDATE SET *

        WHEN NOT MATCHED THEN
        INSERT *
    """)

StatementMeta(, 7e76f964-dd43-4a1f-87fc-8e380ab5258e, 5, Finished, Available, Finished, False)